# SNOM Raster Scan
2-D tip scan with per-point optical and mechanical amplitude + phase readout for harmonics 0–5.

## 0 — Configuration

In [ ]:
HOST        = 'nea-server'
PATH_TO_DLL = ''
SPEED_UM_S  = 0.2          # tip travel speed µm/s

# Scan grid (all in µm)
X_START_UM = 50.0
Y_START_UM = 50.0
X_STEP_UM  = 0.1           # 100 nm pitch
Y_STEP_UM  = 0.1
NX         = 10
NY         = 10
SNAKE      = True          # serpentine scan order

# Sampling
T_INTEG_S  = 0.1           # integration time per point (s)
N_AVG      = 1             # number of integration windows averaged per point

## 1 — Imports

In [ ]:
import asyncio
import time
from time import sleep

import nest_asyncio
import numpy as np
import matplotlib.pyplot as plt
import nea_tools

print('nea_tools version:', getattr(nea_tools, '__version__', 'unknown'))

## 2 — Connect to nea-server

In [ ]:
loop = asyncio.get_event_loop()
nest_asyncio.apply(loop)

print(f'Connecting to {HOST} ...')
loop.run_until_complete(nea_tools.connect(HOST, fingerprint=None, path_to_dll=PATH_TO_DLL))
print('Connected.')

## 3 — Post-connect imports (injected by nea_tools)

In [ ]:
from neaspec import context
import neaspec.stream as stream
import Nea.Client.SharedDefinitions as nea

print('context:', context)
print('nea    :', nea)

## 4 — Move-tip helper

In [ ]:
do_wait = False

def on_tip_position_moved(sender, args):
    print(f'Arrived at {args.Point}')
    global do_wait
    do_wait = False

def on_tip_position_moving(sender, args):
    print(f'Moving to {args.Point}')
    global do_wait
    do_wait = True

context.Logic.TipPositionMoved  += on_tip_position_moved
context.Logic.TipPositionMoving += on_tip_position_moving

def move_tip(x_um: float, y_um: float, speed_um_s: float = SPEED_UM_S) -> None:
    """Move tip to (x_um, y_um) and block until done."""
    move_args = nea.MoveTipPositionArgs(
        nea.Geometry.Point2D(x_um, y_um),
        speed_um_s / 1000.0,
    )
    context.Logic.MoveTipPosition.Execute(move_args)
    sleep(0.1)
    while do_wait:
        sleep(0.1)
    sleep(0.2)

print('move_tip() defined.')

## 5 — Per-point sampler
Polls amplitudes for `T_INTEG_S` seconds and averages, then opens a `Stream()` for phase tail. Repeats `N_AVG` times and returns the mean.

In [ ]:
def sample_point(t_integ_s: float, n_avg: int):
    """Return (o_amp[6], o_phase[6], m_amp[6], m_phase[6]) averaged over n_avg windows."""
    o_amp   = np.zeros(6)
    m_amp   = np.zeros(6)
    o_phase = np.zeros(6)
    m_phase = np.zeros(6)

    for _ in range(n_avg):
        with stream.Stream() as s:
            # amplitudes: poll as fast as possible for t_integ_s seconds
            oa_acc = np.zeros(6)
            ma_acc = np.zeros(6)
            count  = 0
            t_end  = time.time() + t_integ_s
            while time.time() < t_end:
                for h in range(6):
                    oa_acc[h] += context.Microscope.Py.OpticalAmplitude[h]
                    ma_acc[h] += context.Microscope.Py.MechanicalAmplitude[h]
                count += 1
            o_amp += oa_acc / max(count, 1)
            m_amp += ma_acc / max(count, 1)

            # phases: read tail from the same stream
            for h in range(6):
                stream_key = f'O{h}P'
                try:
                    o_phase[h] += s.data[stream_key][-1]
                except Exception as e:
                    o_phase[h] += np.nan
                    print(f'Warning: could not read phase for {stream_key}: {e}')
                stream_key = f'M{h}P'
                try:
                    m_phase[h] += s.data[stream_key][-1]
                except Exception as e:
                    m_phase[h] += np.nan
                    print(f'Warning: could not read phase for {stream_key}: {e}')

    return o_amp / n_avg, o_phase / n_avg, m_amp / n_avg, m_phase / n_avg

print('sample_point() defined.')

## 6 — Allocate maps and define grid iterator

In [ ]:
o_amp_map   = np.full((6, NY, NX), np.nan)
o_phase_map = np.full((6, NY, NX), np.nan)
m_amp_map   = np.full((6, NY, NX), np.nan)
m_phase_map = np.full((6, NY, NX), np.nan)

def grid_points():
    for iy in range(NY):
        xs = range(NX - 1, -1, -1) if (SNAKE and iy % 2 == 1) else range(NX)
        for ix in xs:
            yield ix, iy, X_START_UM + ix * X_STEP_UM, Y_START_UM + iy * Y_STEP_UM

print(f'Grid: {NX}x{NY} = {NX*NY} points  |  '
      f'step ({X_STEP_UM*1000:.0f}, {Y_STEP_UM*1000:.0f}) nm  |  '
      f'integ {T_INTEG_S}s x{N_AVG}')

## 7 — Scan

In [ ]:
total = NX * NY
t0    = time.time()

for k, (ix, iy, x_um, y_um) in enumerate(grid_points(), 1):
    move_tip(x_um, y_um)
    oa, op, ma, mp = sample_point(T_INTEG_S, N_AVG)
    o_amp_map  [:, iy, ix] = oa
    o_phase_map[:, iy, ix] = op
    m_amp_map  [:, iy, ix] = ma
    m_phase_map[:, iy, ix] = mp

    if k % max(total // 20, 1) == 0 or k == total:
        elapsed = time.time() - t0
        eta     = elapsed / k * (total - k)
        print(f'  {k:>4}/{total}  ({100*k/total:.0f}%)  '
              f'elapsed {elapsed:.0f}s  ETA {eta:.0f}s')

print('\nScan complete.')

## 8 — Plot maps

In [ ]:
def plot_maps(maps, title, cmap='viridis', units=''):
    extent = [
        X_START_UM, X_START_UM + NX * X_STEP_UM,
        Y_START_UM, Y_START_UM + NY * Y_STEP_UM,
    ]
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    for h, ax in enumerate(axes.flat):
        im = ax.imshow(
            maps[h], origin='lower', extent=extent,
            cmap=cmap, aspect='equal',
        )
        ax.set_title(f'h = {h}')
        ax.set_xlabel('X (µm)')
        ax.set_ylabel('Y (µm)')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(title + (f'  [{units}]' if units else ''), fontsize=14)
    fig.tight_layout()
    plt.show()

plot_maps(o_amp_map,   'Optical amplitude  O0–O5')
plot_maps(o_phase_map, 'Optical phase  O0–O5',      cmap='twilight', units='rad')
plot_maps(m_amp_map,   'Mechanical amplitude  M0–M5')
plot_maps(m_phase_map, 'Mechanical phase  M0–M5',   cmap='twilight', units='rad')

## 9 — Disconnect

In [ ]:
context.Logic.TipPositionMoved  -= on_tip_position_moved
context.Logic.TipPositionMoving -= on_tip_position_moving

try:
    nea_tools.disconnect()
    print('Disconnected.')
except Exception as e:
    print(f'Disconnect error (ignored): {e}')